In [62]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [63]:
train = pd.read_csv(
    '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
)

test = pd.read_csv(
    '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'
)

In [64]:
print(train.shape)
print(test.shape)

train.columns

(1460, 81)
(1459, 80)


Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [65]:
X = train.drop(columns = 'SalePrice')
y = train['SalePrice']

print(X.shape)
print(y.shape)

(1460, 80)
(1460,)


In [66]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
)

print(X_train.shape)
print(X_valid.shape)
print(y_train.shape)
print(y_valid.shape)

(1168, 80)
(292, 80)
(1168,)
(292,)


In [67]:
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(exclude=['object']).columns

print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))

Categorical columns: 43
Numerical columns: 37


In [68]:
numerical_transformer = SimpleImputer(
    strategy = 'median',
)

In [69]:
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

In [70]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [71]:
y_train_log = np.log1p(y_train)

In [72]:
gbr = GradientBoostingRegressor(
    random_state = 42,
    n_estimators = 640,
    learning_rate = 0.05,
    max_depth = 3,
    subsample = 0.9,
    min_samples_leaf = 4,
)

In [73]:
model = Pipeline(
        steps = [("preprocessor", preprocessor),
                ("regressor", gbr),
                ]
)

In [74]:
model.fit(
    X_train,
    y_train_log,
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath...
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object'))])),
                ('regressor',
                 GradientBoostingRegressor(learning_rate=0.05,
                                           min_samples_leaf=5, n_estimators=640,
                                           random_state=42, subsample=0.9))])

In [75]:
pred = model.predict(X_valid)

score = np.sqrt(
    mean_squared_error(
        np.log1p(y_valid),
        pred,
    )
)

print("Without log(target):", 0.1535781039872966)
print("With log(target):", 0.14637433145697762)
print(f"gbr: {score}")

Without log(target): 0.1535781039872966
With log(target): 0.14637433145697762
gbr: 0.13512786707545485


**下面这里是要看一下不同的训练测试集计算出的score**

In [76]:
kf = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42,
)

cv_scores = abs(cross_val_score(
    model,
    X,
    np.log1p(y),
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
))

print(cv_scores)
print(cv_scores.mean())
print(cv_scores.std())

[0.13683504 0.11182111 0.1700155  0.13015207 0.10621833]
0.1310084074239492
0.022534136984856117
